# Amazon Bedrock Knowledge Base RAG Q&A

`requirements.md` / `design.md` 기준 구현. Retrieve API 기반 Custom RAG Workflow.

- **boto3 방식**: `bedrock-agent-runtime.retrieve` (HYBRID, 5개) + `amazon.nova-lite-v1:0`
- **LangChain 방식**: `AmazonKnowledgeBasesRetriever` (SEMANTIC, 4개) + PromptTemplate + LCEL + ChatBedrock + StrOutputParser

흐름: 사용자 입력 → Retrieve API(쿼리 임베딩·유사 문서 검색) → 컨텍스트 → 프롬프트 증강 → LLM → 답변

## 0. 의존성 설치 (최초 1회)

이미 설치되어 있으면 건너뛰어도 됩니다. `pip`는 멱등적으로 동작하므로 중복 실행해도 안전합니다.

In [ ]:
# %pip install -q boto3 langchain langchain-aws

## 1. 공통 설정

리전과 모델 ID, 검증 질문을 정의합니다. (design.md 9. 설정 항목)

In [ ]:
import json
import boto3

# 리전은 환경/프로파일 기본값을 우선 사용하되, 없으면 기본 리전 지정
REGION = boto3.session.Session().region_name or "us-east-1"
MODEL_ID = "amazon.nova-lite-v1:0"

# 검증 질문 (requirements.md)
QUESTION = (
    "What was the total operating lease liabilities and total sublease income "
    "of the AnyCompany as of December 31, 2022?"
)

print("Region:", REGION)
print("Model :", MODEL_ID)

## 2. Knowledge Base ID 조회 (R1 / D4)

기존 Knowledge Base만 조회한다. 새로 생성하지 않으며(멱등성), 없으면 명확히 오류를 발생시킨다.

In [ ]:
def get_knowledge_base_id(region: str = REGION) -> str:
    """기존 Knowledge Base 목록에서 첫 번째 KB ID를 반환한다. (생성하지 않음)"""
    agent = boto3.client("bedrock-agent", region_name=region)
    paginator = agent.get_paginator("list_knowledge_bases")
    for page in paginator.paginate():
        for kb in page.get("knowledgeBaseSummaries", []):
            return kb["knowledgeBaseId"]
    raise RuntimeError(
        "사용 가능한 Knowledge Base가 없습니다. 먼저 Bedrock Knowledge Base를 생성하세요."
    )


KB_ID = get_knowledge_base_id()
print("Knowledge Base ID:", KB_ID)

## 3. boto3 Retrieve 방식

### 3.1 Retrieve (HYBRID, 5개) + content.text 추출 (R2, R3 / D5.2, D5.3)

In [ ]:
def retrieve_boto3(kb_id: str, query: str, num_results: int = 5, region: str = REGION):
    """Retrieve API로 KB를 검색한다. 검색 방식 HYBRID, 기본 5개."""
    runtime = boto3.client("bedrock-agent-runtime", region_name=region)
    resp = runtime.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "numberOfResults": num_results,
                "overrideSearchType": "HYBRID",
            }
        },
    )
    return resp.get("retrievalResults", [])


def build_context(results) -> str:
    """검색 결과에서 content.text 를 추출하여 하나의 context 문자열로 결합."""
    return "\n\n".join(
        r["content"]["text"]
        for r in results
        if r.get("content", {}).get("text")
    )


results = retrieve_boto3(KB_ID, QUESTION)
context = build_context(results)
print(f"검색 결과 {len(results)}개\n")
print(context[:1000], "..." if len(context) > 1000 else "")

### 3.2 프롬프트 증강 (환각 방지) (R4 / D5.4)

In [ ]:
PROMPT_TEMPLATE = """당신은 제공된 컨텍스트만을 근거로 답변하는 어시스턴트입니다.
컨텍스트에 없는 내용은 추측하지 말고 "정보를 찾을 수 없습니다"라고 답하세요.

<context>
{context}
</context>

질문: {question}
답변:"""


def build_prompt(context: str, question: str) -> str:
    return PROMPT_TEMPLATE.format(context=context, question=question)


prompt = build_prompt(context, QUESTION)
print(prompt[:800], "...")

### 3.3 답변 생성 - amazon.nova-lite-v1:0 (R5 / D5.5)

In [ ]:
def generate_answer_boto3(prompt: str, region: str = REGION) -> str:
    """amazon.nova-lite-v1:0 을 invoke_model 로 호출하여 답변 생성."""
    runtime = boto3.client("bedrock-runtime", region_name=region)
    body = {
        "messages": [
            {"role": "user", "content": [{"text": prompt}]}
        ],
        "inferenceConfig": {"maxTokens": 512, "temperature": 0.0},
    }
    resp = runtime.invoke_model(modelId=MODEL_ID, body=json.dumps(body))
    payload = json.loads(resp["body"].read())
    return payload["output"]["message"]["content"][0]["text"]


answer_boto3 = generate_answer_boto3(prompt)
print("=== boto3 방식 답변 ===")
print(answer_boto3)

## 4. LangChain LCEL 방식 (R6 / D6)

AmazonKnowledgeBasesRetriever(SEMANTIC, 4개) + PromptTemplate + LCEL + ChatBedrock + StrOutputParser

In [ ]:
from langchain_aws import AmazonKnowledgeBasesRetriever, ChatBedrock
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs) -> str:
    return "\n\n".join(d.page_content for d in docs)


def build_langchain_chain(kb_id: str, region: str = REGION):
    retriever = AmazonKnowledgeBasesRetriever(
        knowledge_base_id=kb_id,
        region_name=region,
        retrieval_config={
            "vectorSearchConfiguration": {
                "numberOfResults": 4,
                "overrideSearchType": "SEMANTIC",
            }
        },
    )

    prompt = PromptTemplate(
        template=PROMPT_TEMPLATE,
        input_variables=["context", "question"],
    )

    llm = ChatBedrock(
        model_id=MODEL_ID,
        region_name=region,
        model_kwargs={"temperature": 0.0, "max_tokens": 512},
    )

    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


chain = build_langchain_chain(KB_ID)
answer_langchain = chain.invoke(QUESTION)
print("=== LangChain LCEL 방식 답변 ===")
print(answer_langchain)

## 5. 결과 비교 (D7 / 완료 조건)

In [ ]:
print("질문:", QUESTION)
print("\n--- boto3 (HYBRID, 5) ---")
print(answer_boto3)
print("\n--- LangChain LCEL (SEMANTIC, 4) ---")
print(answer_langchain)